# R10-H100 - structural profiles predict probe fate: ROMEO re-derived for retrieval

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-07 <br>
**Pipeline stage**: R10 graph-topology round <br>
**Graph**: rebuilt CPAP graph (neo4j2, read-only), Titan embeddings, zero completions <br>

Task-based ontology evaluation derives task-specific structural metrics whose profiles predict task
performance. The metric derivation transfers; the values do not - we re-derive against probe answering.

## Approach (as registered)
1. **Touched subgraph** per probe = vec-8 seeds + 1-hop neighbours (valid non-SIMILAR_TO edges)
2. **Profile features** - type population, average connectivity, relationship-type diversity,
   proposition density (per the registration)
3. **Target** - per-probe DIRECT-RENDER gold hit rate: fraction of that probe's golds present in the
   seed-render context (varies unlike full recall 1.0). The per-gold binary presence-in-direct-render
   is the classification unit; each gold inherits its probe's structural profile
4. **Fit** - logistic regression, leave-one-probe-out CV, ROC AUC on held-out predictions; plus per-probe
   Spearman rank correlations of each feature vs the probe hit rate
5. **Verdict** - AUC > 0.7 confirms; no profile correlates refutes (extraction fidelity dominates)

## Outputs
- `reports/structural-profiles-h100-<stamp>.json`


In [1]:
# Imports
# stdlib
import datetime, json, os, re
from pathlib import Path
# third party
import numpy as np
import yaml
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from rich import print as rprint
from rich.progress import Progress

os.environ["NEO4J_URI"] = "bolt://user-konrad.jelen-kgf-neo4j2:7687"   # read-only neo4j2
os.environ["NEO4J_USER"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "kgfoundry"

from knowledge_graph_foundry import Foundry, load_settings
from knowledge_graph_foundry.extraction import generate_embeddings
from knowledge_graph_foundry.graph.graphrag import vector_query
from knowledge_graph_foundry.models import Entity

PROBES_PATH = Path("../tests/probes/cpap-probe-set.yml")
settings = load_settings(Path("../config.yml"))
TOP_K = settings.graphrag.top_k                # vec-8
VEC_INDEX = settings.graphrag.vector_index_name
FEATURES = ["type_population", "avg_connectivity", "rel_type_diversity", "proposition_density"]
BAR_AUC = 0.70

probes = yaml.safe_load(PROBES_PATH.read_text())
gold_probes = [p for p in probes if p.get("gold_evidence")]
rprint(f"[bold]config[/bold] vec_top_k={TOP_K}  features={FEATURES}  bar AUC>{BAR_AUC}  "
       f"gold_probes={len(gold_probes)}")


2026-07-07 12:20:07.711 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


config vec_top_k=8  features=['type_population', 'avg_connectivity', 'rel_type_diversity', 'proposition_density']  
bar AUC>0.7  gold_probes=24

## Graph snapshot: entities, edges, proposition counts\n\nOne pass loads the valid non-SIMILAR_TO adjacency (with edge relation types), entity type labels, direct-render surfaces, and attached-proposition counts. The direct-render surface replicates the vec-channel node render (name + description + prop_* spec + relation lines) used for the DIRECT-RENDER target.

In [2]:
def _norm(s):
    return re.sub(r"\s+", " ", s.casefold())

with Foundry(settings) as f:
    with f.driver.session() as s:
        ents = s.run("MATCH (e:Entity) RETURN e.id AS id, e.name AS name, e.description AS description, "
                     "labels(e) AS types, properties(e) AS props").data()
        edges = s.run("MATCH (a:Entity)-[r]-(b:Entity) "
                      "WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' AND a.id < b.id "
                      "RETURN DISTINCT a.id AS a, b.id AS b, type(r) AS rel").data()
        prop_counts = s.run("MATCH (p:Proposition)-[:ABOUT]->(e:Entity) "
                            "RETURN e.id AS eid, count(p) AS n").data()

types = {r["id"]: [t for t in r["types"] if t != "Entity"] for r in ents}
propn = {r["eid"]: r["n"] for r in prop_counts}
names = {r["id"]: r["name"] for r in ents}

# adjacency + incident relation types + global degree
adj = {}
rel_at = {}     # node -> list of incident rel types
for e_ in edges:
    adj.setdefault(e_["a"], set()).add(e_["b"])
    adj.setdefault(e_["b"], set()).add(e_["a"])
    rel_at.setdefault(e_["a"], []).append(e_["rel"])
    rel_at.setdefault(e_["b"], []).append(e_["rel"])
degree = {nid: len(adj.get(nid, ())) for nid in names}

# direct-render surface per entity (vec channel node render, no alias/prop)
surface = {}
for r in ents:
    spec = {k.removeprefix("prop_"): v for k, v in r["props"].items() if k.startswith("prop_")}
    rels = rel_at.get(r["id"], [])
    surface[r["id"]] = _norm(" ".join([
        r["name"] or "", r["description"] or "", json.dumps(spec, default=str),
        " ".join(f"{rl} {names.get(nb,'')}" for rl, nb in
                 [(e_["rel"], e_["b"] if e_["a"] == r["id"] else e_["a"])
                  for e_ in edges if r["id"] in (e_["a"], e_["b"])]),
    ]))
rprint(f"entities [yellow]{len(ents)}[/yellow]  edges [yellow]{len(edges)}[/yellow]  "
       f"entities-with-props [yellow]{len(propn)}[/yellow]")


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=41, offset=40>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 40, 'line': 1, 'column': 41}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (a:Entity)-[r]-(b:Entity) WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' AND a.id < b.id RETURN DISTINCT a.id AS a, b.id AS b, type(r) AS rel"


entities 2798  edges 3905  entities-with-props 2759

## Matcher (H34 verbatim)

In [3]:
def value_tokens(text):
    return re.findall(r"[\w.\-/]*\d[\w.\-/]*", text)

_UNIT = r"(?<=\d)\s*(mm|cm|dba|db\(a\)|db|kg|g|oz|ml|l|w|hz|mins|min|m)\b"

def present(gold, ctx_norm):
    ng = _norm(gold)
    if ng in ctx_norm:
        return True
    squashed = re.sub(r"[\s,()]", "", ctx_norm)
    skeleton = re.sub(r"[\s,()]", "", re.sub(_UNIT, "", ng))
    if any(ch.isdigit() for ch in skeleton) and len(skeleton) >= 5 and skeleton in squashed:
        return True
    tokens = value_tokens(gold)
    if tokens:
        hit = sum(1 for t in tokens if _norm(t) in ctx_norm or re.sub(r"[\s,()]", "", _norm(t)) in squashed)
        return hit >= max(1, len(tokens) // 2 + (len(tokens) % 2))
    words = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ng))
    ctx_words = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ctx_norm))
    return bool(words) and len(words & ctx_words) / len(words) >= 0.6


## Per-probe profile + direct-render target\n\nPer probe: Titan embedding -> vec-8 seeds -> touched subgraph (seeds + 1-hop). Profile features computed over the subgraph. Target: each gold's presence in the direct-render context (the concatenated vec-8 seed surfaces).

In [4]:
rows = []      # per gold
prof_rows = [] # per probe
with Foundry(settings) as f, Progress() as pr:
    t = pr.add_task("profile", total=len(gold_probes))
    for p in gold_probes:
        q = p["question"]
        probe_e = Entity.create(q[:80], types=["Query"], description=q)
        emb = generate_embeddings([probe_e], settings.embeddings)[0].embedding
        seeds = [s["id"] for s in vector_query(f.driver, emb, VEC_INDEX, top_k=TOP_K)]
        seeds = [s for s in seeds if s in names]
        # touched subgraph = seeds + 1-hop
        sub = set(seeds)
        for sd in seeds:
            sub |= adj.get(sd, set())
        sub = [n for n in sub if n in names]
        # features
        type_pop = len({tp for n in sub for tp in types.get(n, [])})
        avg_conn = float(np.mean([degree[n] for n in sub])) if sub else 0.0
        rel_div = len({e_["rel"] for e_ in edges if e_["a"] in sub or e_["b"] in sub})
        prop_den = float(np.mean([propn.get(n, 0) for n in sub])) if sub else 0.0
        feat = {"type_population": type_pop, "avg_connectivity": avg_conn,
                "rel_type_diversity": rel_div, "proposition_density": prop_den}
        # direct-render context = concatenation of seed surfaces
        ctx = _norm(" ".join(surface[s] for s in seeds))
        golds = p["gold_evidence"]
        hits = [1 if present(g, ctx) else 0 for g in golds]
        hit_rate = float(np.mean(hits)) if golds else 0.0
        prof_rows.append({"probe": p["id"], **feat, "n_golds": len(golds),
                          "n_subgraph": len(sub), "direct_hit_rate": hit_rate})
        for g, h in zip(golds, hits):
            rows.append({"probe": p["id"], "gold": g[:60], "y": h, **feat})
        pr.advance(t)

rprint(f"[green]profiled[/green] {len(prof_rows)} probes  {len(rows)} golds")
rprint(f"mean direct-render hit rate: [yellow]{np.mean([r['direct_hit_rate'] for r in prof_rows]):.3f}[/yellow] "
       f"[dim](variance target - full recall would be 1.0)[/dim]")


/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

2026-07-07 12:20:13.050 | INFO     | knowledge_graph_foundry.extraction.embeddings:_embed_bedrock:91 - Embeddings: 
1/1

2026-07-07 12:20:13.052 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - 
Generated embeddings for 1/1 entities via bedrock (0 cache hits)

2026-07-07 12:20:13.312 | INFO     | knowledge_graph_foundry.extraction.embeddings:_embed_bedrock:91 - Embeddings: 
1/1

2026-07-07 12:20:13.314 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - 
Generated embeddings for 1/1 entities via bedrock (0 cache hits)

...

2026-07-07 12:20:18.911 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - 
Generated embeddings for 1/1 entities via bedrock (0 cache hits)

2026-07-07 12:20:19.194 | INFO     | knowledge_graph_foundry.extraction.embeddings:_embed_bedrock:91 - Embeddings: 
1/1

2026-07-07 12:20:19.197 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - 
Generated embeddings for 1/1 entities via bedrock (0 cache hits)

profiled 24 probes  33 golds

mean direct-render hit rate: 0.667 (variance target - full recall would be 1.0)

## Logistic fit (leave-one-probe-out) + rank correlations\n\nPer-gold binary presence in the direct render, features = the probe's structural profile. Leave-one-probe-out CV (all golds of the held-out probe scored by a model fit on the rest) yields held-out probabilities; ROC AUC over all golds. Univariate Spearman correlations (per-gold and per-probe) test which features carry signal.

In [5]:
X = np.array([[r[f] for f in FEATURES] for r in rows], dtype=float)
y = np.array([r["y"] for r in rows])
probe_ids = np.array([r["probe"] for r in rows])
uprobes = list(dict.fromkeys(probe_ids))

rprint(f"class balance: present={int(y.sum())}  absent={int((1-y).sum())}  of {len(y)}")

# leave-one-probe-out predicted probabilities
oof = np.full(len(y), np.nan)
if 0 < y.sum() < len(y):
    for hp in uprobes:
        te = probe_ids == hp
        tr = ~te
        if len(set(y[tr])) < 2:
            continue
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(max_iter=1000, C=1.0).fit(sc.transform(X[tr]), y[tr])
        oof[te] = clf.predict_proba(sc.transform(X[te]))[:, 1]
mask = ~np.isnan(oof)
auc = roc_auc_score(y[mask], oof[mask]) if len(set(y[mask])) == 2 else float("nan")

# full-fit coefficients (standardized) for interpretation
scaler = StandardScaler().fit(X)
full = LogisticRegression(max_iter=1000, C=1.0).fit(scaler.transform(X), y)
coefs = dict(zip(FEATURES, full.coef_[0].round(4)))

# univariate spearman (per gold) and per-probe (feature vs hit rate)
pg_sp = {f: spearmanr(X[:, i], y).correlation for i, f in enumerate(FEATURES)}
pv = {f: [r[f] for r in prof_rows] for f in FEATURES}
hr = [r["direct_hit_rate"] for r in prof_rows]
pp_sp = {f: spearmanr(pv[f], hr).correlation for f in FEATURES}

rprint(f"""[bold cyan]Structural profiles -> direct-render fate[/bold cyan]
[dim]{"-"*44}[/dim]
  Leave-one-probe-out ROC AUC (per gold): [bold yellow]{auc:.3f}[/bold yellow] [dim](bar > {BAR_AUC})[/dim]
  Standardized logistic coefficients: {coefs}
  Per-gold Spearman (feature vs present): {{ {', '.join(f'{f}:{pg_sp[f]:+.2f}' for f in FEATURES)} }}
  Per-probe Spearman (feature vs hit rate): {{ {', '.join(f'{f}:{pp_sp[f]:+.2f}' for f in FEATURES)} }}
""")

verdict = ("CONFIRMED" if (auc == auc and auc > BAR_AUC) else "REFUTED")
rprint(f"  Verdict: [{'green' if verdict=='CONFIRMED' else 'red'}]{verdict}[/]  "
       f"[dim]{'a structural profile predicts direct-render fate' if verdict=='CONFIRMED' else 'no profile clears the bar - extraction fidelity dominates structure (H51 gains indirect support)'}[/dim]")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
out = Path("../reports") / f"structural-profiles-h100-{stamp}.json"
out.write_text(json.dumps({
    "hypothesis": "R10-H100", "features": FEATURES, "unit": "per-gold binary direct-render presence",
    "n_probes": len(prof_rows), "n_golds": len(rows),
    "mean_direct_hit_rate": float(np.mean(hr)),
    "loo_auc": auc, "bar_auc": BAR_AUC,
    "logistic_coefficients_standardized": coefs,
    "spearman_per_gold": pg_sp, "spearman_per_probe": pp_sp,
    "verdict": verdict,
    "profiles": prof_rows,
}, indent=2, default=str))
rprint("saved", str(out))


class balance: present=21  absent=12  of 33

Structural profiles -> direct-render fate
--------------------------------------------
  Leave-one-probe-out ROC AUC (per gold): 0.496 (bar > 0.7)
  Standardized logistic coefficients: {'type_population': -0.0523, 'avg_connectivity': 0.0483, 
'rel_type_diversity': 0.6248, 'proposition_density': 0.0287}
  Per-gold Spearman (feature vs present): { type_population:+0.15, avg_connectivity:-0.03, 
rel_type_diversity:+0.25, proposition_density:-0.04 }
  Per-probe Spearman (feature vs hit rate): { type_population:+0.12, avg_connectivity:+0.08, 
rel_type_diversity:+0.37, proposition_density:+0.08 }

Verdict: REFUTED  no profile clears the bar - extraction fidelity dominates structure (H51 gains indirect 
support)

saved ../reports/structural-profiles-h100-20260707-102019.json